In [9]:
import pandas as pd
import joblib
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score

# Load datasets
capsicum_df = pd.read_csv("capsicum.csv")
tomato_df = pd.read_csv("tomato.csv")

# Function to preprocess data & train yield prediction model
def train_yield_model(df, crop_name):
    X = df[["Area"]]
    y = df["Crop_Yield"]

    # Train-Test Split
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    # Standardize Area
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    # Train Model
    model = RandomForestRegressor(n_estimators=200, random_state=42)
    model.fit(X_train_scaled, y_train)

    # Evaluate Model
    y_pred = model.predict(X_test_scaled)
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)

    print(f"✅ {crop_name} Model Performance:")
    print(f"   - Mean Absolute Error (MAE): {mae:.2f}")
    print(f"   - R² Score: {r2:.2f}")

    # Save Model & Scaler
    joblib.dump(model, f"{crop_name}_yield_model.pkl")
    joblib.dump(scaler, f"{crop_name}_scaler.pkl")

    print(f"✅ {crop_name} Yield Model & Scaler Saved!\n")

# Train & Save Models
train_yield_model(tomato_df, "tomato")
train_yield_model(capsicum_df, "capsicum")

print("✅ All models trained & evaluated!")

# -------------------------------------------
# 🔹 TESTING PART (Yield Prediction Testing)
# -------------------------------------------
def test_yield_prediction(crop_name, area_value):
    try:
        # Load appropriate model & scaler
        model = joblib.load(f"{crop_name}_yield_model.pkl")
        scaler = joblib.load(f"{crop_name}_scaler.pkl")

        # Convert input to DataFrame (Fix for warning)
        area_df = pd.DataFrame([[area_value]], columns=["Area"])
        area_scaled = scaler.transform(area_df)

        # Predict yield
        predicted_yield = model.predict(area_scaled)[0]

        print(f"🌾 Predicted {crop_name} Yield for Area {area_value} ha: {predicted_yield:.2f} metric tons")

    except Exception as e:
        print(f"❌ Error in prediction: {e}")


# 🔍 Example Test Cases
test_yield_prediction("tomato", 1.2)
test_yield_prediction("capsicum", 0.5)


✅ tomato Model Performance:
   - Mean Absolute Error (MAE): 0.00
   - R² Score: 1.00
✅ tomato Yield Model & Scaler Saved!

✅ capsicum Model Performance:
   - Mean Absolute Error (MAE): 0.00
   - R² Score: 1.00
✅ capsicum Yield Model & Scaler Saved!

✅ All models trained & evaluated!
🌾 Predicted tomato Yield for Area 1.2 ha: 13.49 metric tons
🌾 Predicted capsicum Yield for Area 0.5 ha: 2.95 metric tons
